# Необходимые библиотекы для работы 

In [2]:
# базовые библиотеки, модули
import pandas as pd
import numpy as np

#визуализация
import matplotlib as plt
import seaborn as sns

# Загрузка данных

In [5]:
# обучающая выборка
train_df = pd.read_csv("I'll come back later data/train.csv")

In [6]:
# тестовая выборка
test_df = pd.read_csv("I'll come back later data/test.csv")

# EDA

In [7]:
# класс - ускоритель для сбора первичной информации о датасете
class DatasetReview:    
    def __init__(self,dataframe):
        self.df = dataframe
        self.num_feats = self.df.select_dtypes(include=["number"]).columns
        self.cat_feats = self.df.select_dtypes(include=["object","category","bool"]).columns
        
    def collect_min_info(self):
        print("-" * 10 + "Первые 10 строк датасета" + "-" * 10 + "\n")
        display(self.df.head(10))
        print("\n"+"-" * 10 + "Общие сведения" + "-" * 10 + "\n")
        #определяем признаки
        df_info = pd.DataFrame({
            "Columns":self.df.columns,
            "Тип": self.df.dtypes.values,
            "Уникальные значения":self.df.nunique().values,
            "Заполненные значения (кол-во) ":self.df.count().values,
            "Пропущенные значения (кол-во) ":self.df.isnull().sum().values,
            "Пропущенные значения (проценты) ": (self.df.isnull().sum().values / len(self.df) * 100).round(2),
        })
        with pd.option_context('display.expand_frame_repr', False):
             display(df_info)
        print(f"{self.df.shape} - количество строк/столбцов")
        
        print("\n"+"-" * 10 + "Оценка значений(числовые,бинарные признаки)" + "-" * 10 + "\n")
        if not self.num_feats.empty:
            display(round(self.df[self.num_feats].describe(),2))

        print("\n"+"-" * 10 + "Оценка значений(категориальные признаки)" + "-" * 10 + "\n")
        if not self.cat_feats.empty:
            display(round(self.df[self.cat_feats].describe(),2))
        
        print("\n"+"-" * 10 + "columns" + "-" * 10 + "\n")
        columns = self.df.columns.to_list()
        print(f"ОБЩИЙ СПИСОК ПРИЗНАКОВ:{columns}")

    def get_null(self) -> pd.DataFrame:
        null_rows = self.df[self.df.isnull().any(axis=1)]
        return null_rows

    def get_duplicates(self,cols:list = None,keep="first") -> pd.DataFrame:
        duplicates_mask = self.df.duplicated(keep=keep,subset=cols)
        duplicated_df = self.df[duplicates_mask]
        return duplicated_df

    def get_numeric_feats(self) -> list[str]:
        return self.num_feats.tolist()

    def get_categorical_feats(self) -> list[str]:
        return self.cat_feats.tolist()

In [8]:
reviewer = DatasetReview(train_df)
reviewer.collect_min_info()

----------Первые 10 строк датасета----------



,id,sessions_count,avg_session_time,days_since_last_activity,purchases_count,avg_purchase_value,active_days,session_std,is_weekend_user,retention
0,1045,16,46.079173,9,13,5462.010418,18,14.307116,1,0
1,1339,1,42.625268,24,8,4679.746955,14,26.690708,1,1
2,5144,16,67.310472,21,12,6532.157783,21,13.092315,1,1
3,7099,16,78.904027,34,14,7155.049697,28,10.564979,0,0
4,7907,8,50.456513,19,12,6214.569079,25,6.813272,1,0
5,8270,8,53.347437,13,15,5488.464653,24,14.155992,0,1
6,7561,13,61.133285,33,13,6048.712739,19,12.939032,0,1
7,7178,1,46.142219,5,10,4391.536260,19,20.047936,0,0
8,1164,12,56.564963,10,14,5862.348197,20,13.027084,0,0
9,6735,1,42.455049,24,8,4365.236948,15,13.502038,0,0



----------Общие сведения----------



,Columns,Тип,Уникальные значения,Заполненные значения (кол-во),Пропущенные значения (кол-во),Пропущенные значения (проценты)
0,id,int64,7595,7595,0,0.0
1,sessions_count,int64,31,7595,0,0.0
2,avg_session_time,float64,7588,7595,0,0.0
3,days_since_last_activity,int64,55,7595,0,0.0
4,purchases_count,int64,19,7595,0,0.0
5,avg_purchase_value,float64,7444,7595,0,0.0
6,active_days,int64,23,7595,0,0.0
7,session_std,float64,7574,7595,0,0.0
8,is_weekend_user,int64,2,7595,0,0.0
9,retention,int64,2,7595,0,0.0


(7595, 10) - количество строк/столбцов

----------Оценка значений(числовые,бинарные признаки)----------



,id,sessions_count,avg_session_time,days_since_last_activity,purchases_count,avg_purchase_value,active_days,session_std,is_weekend_user,retention
count,7595.00,7595.00,7595.00,7595.00,7595.00,7595.00,7595.00,7595.00,7595.0,7595.00
mean,5042.75,10.43,56.82,19.57,12.15,5870.97,20.23,11.91,0.5,0.73
std,2937.71,5.64,12.01,9.33,2.44,1429.78,4.14,3.76,0.5,0.44
min,1.00,1.00,23.73,0.00,3.00,2097.95,8.00,0.00,0.0,0.00
25%,2461.50,7.00,47.16,13.00,11.00,4780.69,17.00,9.50,0.0,0.00
50%,5042.00,12.00,54.47,20.00,12.00,5542.28,20.00,11.79,0.0,1.00
75%,7580.50,15.00,66.07,26.00,14.00,6880.58,22.00,14.21,1.0,1.00
max,10127.00,31.00,90.00,55.00,21.00,10000.00,30.00,30.66,1.0,1.00



----------Оценка значений(категориальные признаки)----------


----------columns----------

ОБЩИЙ СПИСОК ПРИЗНАКОВ:['id', 'sessions_count', 'avg_session_time', 'days_since_last_activity', 'purchases_count', 'avg_purchase_value', 'active_days', 'session_std', 'is_weekend_user', 'retention']


In [9]:
d = reviewer.get_null()
display(d)

,id,sessions_count,avg_session_time,days_since_last_activity,purchases_count,avg_purchase_value,active_days,session_std,is_weekend_user,retention


In [10]:
c = reviewer.get_duplicates(cols=["avg_session_time"])
display(c)

,id,sessions_count,avg_session_time,days_since_last_activity,purchases_count,avg_purchase_value,active_days,session_std,is_weekend_user,retention
1177,4873,17,90.0,12,14,8260.920762,19,12.270643,0,0
2637,5416,16,90.0,16,15,8688.963639,20,19.875508,0,1
3222,1916,15,90.0,12,13,7970.414161,17,15.353449,0,1
5390,6577,12,90.0,25,14,8080.754299,30,13.816546,1,1
5789,5640,3,90.0,5,16,7831.939530,21,20.299056,0,1
6486,3935,11,90.0,11,17,8458.962412,25,7.073155,0,1
6877,3221,8,90.0,18,17,8058.381533,30,13.508387,1,1


In [11]:
num_cols = reviewer.get_numeric_feats()
print(num_cols)

['id', 'sessions_count', 'avg_session_time', 'days_since_last_activity', 'purchases_count', 'avg_purchase_value', 'active_days', 'session_std', 'is_weekend_user', 'retention']


In [12]:
cat_cols = reviewer.get_categorical_feats()
print(cat_cols)

[]
